# Oruk Fourier 2B — English ASR Benchmark Evaluation

Evaluates [**`oruk/oruk-fourier-2b`**](https://huggingface.co/oruk/oruk-fourier-2b) (pinned BF16 revision `8c2a8871f1304aa2f634da4fb95e24438c4945b9`, `Qwen3ASRForConditionalGeneration`, **CC BY-NC 4.0** non-commercial weights) using the shared [`common.py`](common.py) harness (`read_audio`, `chunk`, `score`, `rollup`).

- **Canonical Hugging Face loading:** downloads native checkpoint files directly from `oruk/oruk-fourier-2b` (`transformers>=5.24.0`).
- **Decode modes:** supports both `published` (default 3-beam search from `generation_config.json`) and `greedy` (`num_beams=1`, `do_sample=False`).
- **Long-form Tier C chunking:** non-overlapping 30 s windows (`chunk(audio, win_s=30, hop_s=30)`) to avoid boundary word duplication.
- **Appends** rows to `results/per_item.csv` with the exact benchmark schema (`model, tier, item, band, audio_s, S, D, I, N, infer_s, vram_gb`).

In [ ]:
# CELL 1 — install dependencies (native Qwen3-ASR requires transformers>=5.24.0)
!pip install -q "transformers>=5.24.0" accelerate soundfile librosa jiwer pandas soxr

In [ ]:
# CELL 2 — mount Google Drive (skip if running outside Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Running outside Google Colab; configure BASE_DIR below.')

In [ ]:
# CELL 3 — configuration & shared harness import
import gc
import json
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForMultimodalLM, AutoProcessor

REPO_DIR = Path('/content/drive/MyDrive/asr_benchmark/english-asr-benchmark')
if REPO_DIR.exists() and str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
elif str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

from common import SR, chunk, read_audio, rollup, score

BASE_DIR = Path('/content/drive/MyDrive/asr_benchmark')
SHORT_AUDIO_DIR = Path('/content/ps/people_speech/peoples_speech_audio')
SHORT_TEXT_DIR = Path('/content/ps/people_speech/peoples_speech_transcriptions')
OUT_DIR = BASE_DIR / 'results'
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_ID = 'oruk/oruk-fourier-2b'
MODEL_REVISION = '8c2a8871f1304aa2f634da4fb95e24438c4945b9'
DECODE_MODE = 'published'  # 'published' (3-beam default) or 'greedy' (1-beam)
SHORT_BATCH_SIZE = 4
LONG_BATCH_SIZE = 4

TIER_B = [
    ('s1', 's1.wav', 's1.txt'),
    ('s5', 's5.wav', 's5.txt'),
]
TIER_C = [
    ('ES2002a', 'ES2002a.Array1-01.wav', 'ES2002a_reference.txt', '16kHz'),
    ('ES2004a', 'ES2004a.Array1-01.wav', 'ES2004a_reference.txt', '16kHz'),
    ('IS1000a', 'IS1000a.Array1-01.wav', 'IS1000a_reference.txt', '16kHz'),
    ('ES2002a_8k', 'ES2002a.Array1-01_8k.wav', 'ES2002a_reference.txt', '8kHz'),
    ('ES2004a_8k', 'ES2004a.Array1-01_8k.wav', 'ES2004a_reference.txt', '8kHz'),
    ('IS1000a_8k', 'IS1000a.Array1-01_8k.wav', 'IS1000a_reference.txt', '8kHz'),
]
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU ONLY')

In [ ]:
# CELL 4 — Oruk Fourier 2B model wrapper conforming to the benchmark REGISTRY contract
class OrukFourier2B:
    key = 'oruk-fourier-2b'

    def __init__(self, decode_mode: str = DECODE_MODE):
        if decode_mode not in {'published', 'greedy'}:
            raise ValueError(f'Unsupported decode_mode={decode_mode!r}')
        self.decode_mode = decode_mode
        self.label = (
            'Oruk-Fourier-2B (3-beam)' if decode_mode == 'published' else 'Oruk-Fourier-2B (greedy)'
        )

    def load(self):
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
        dtype = torch.bfloat16 if (device == 'cuda' and torch.cuda.is_bf16_supported()) else (
            torch.float16 if device == 'cuda' else torch.float32
        )
        self.processor = AutoProcessor.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
        self.processor.tokenizer.padding_side = 'left'
        self.model = (
            AutoModelForMultimodalLM.from_pretrained(
                MODEL_ID,
                revision=MODEL_REVISION,
                dtype=dtype,
            )
            .to(device)
            .eval()
        )
        self.gen_kwargs = {'do_sample': False, 'num_beams': 1} if self.decode_mode == 'greedy' else {}

    def _t(self, arrays, bs):
        out = []
        with torch.inference_mode():
            for i in range(0, len(arrays), bs):
                batch_wavs = arrays[i : i + bs]
                inputs = self.processor.apply_transcription_request(
                    audio=batch_wavs,
                    language=['English'] * len(batch_wavs),
                    return_tensors='pt',
                    processor_kwargs={'padding': True},
                ).to(self.model.device, self.model.dtype)
                ids = self.model.generate(**inputs, **self.gen_kwargs)
                gen = ids[:, inputs['input_ids'].shape[1] :]
                decoded = self.processor.decode(gen, return_format='transcription_only')
                if isinstance(decoded, str):
                    decoded = [decoded]
                out.extend(str(t).strip() for t in decoded)
        return out

    def batch(self, arrays):
        return self._t(arrays, SHORT_BATCH_SIZE)

    def long(self, audio):
        # Non-overlapping 30 s windows prevent boundary word duplication.
        return ' '.join(t for t in self._t(list(chunk(audio, win_s=30, hop_s=30)), LONG_BATCH_SIZE) if t)

In [ ]:
# CELL 5 — discover files, run evaluation, and append to results/per_item.csv
def find_transcript(audio_path: Path):
    stem = audio_path.stem
    for cand in [
        SHORT_TEXT_DIR / f'{stem}.txt',
        SHORT_TEXT_DIR / f'{stem}_flac.txt',
        SHORT_TEXT_DIR / f'{audio_path.name}.txt',
    ]:
        if cand.exists():
            return cand
    return None

def discover_short():
    if not SHORT_AUDIO_DIR.exists():
        return []
    out = []
    for a in sorted(SHORT_AUDIO_DIR.glob('*.flac')) + sorted(SHORT_AUDIO_DIR.glob('*.wav')):
        t = find_transcript(a)
        if t:
            out.append((a.stem, a, t))
    return out

def measure(fn):
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
    t0 = time.perf_counter()
    res = fn()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    el = time.perf_counter() - t0
    vram = torch.cuda.max_memory_allocated() / 1024**3 if torch.cuda.is_available() else 0.0
    return res, el, vram

SHORT_CLIPS = discover_short()
mdl = OrukFourier2B(decode_mode=DECODE_MODE)
mdl.load()
mdl.batch([np.zeros(SR * 5, dtype=np.float32)])  # warmup

rows = []
transcripts = {}

if SHORT_CLIPS:
    arrays = [read_audio(p) for _, p, _ in SHORT_CLIPS]
    total_audio = sum(len(a) for a in arrays) / SR
    hyps, el, vram = measure(lambda: mdl.batch(arrays))
    agg = {'S': 0, 'D': 0, 'I': 0, 'N': 0}
    for (cid, _, tpath), h in zip(SHORT_CLIPS, hyps):
        s, hn = score(tpath.read_text(encoding='utf-8', errors='replace'), h)
        transcripts[f'{mdl.label}|A|{cid}'] = hn
        for k in agg:
            agg[k] += s[k]
    rows.append({
        'model': mdl.label, 'tier': 'A', 'item': f'{len(arrays)} clips',
        'band': '16kHz', 'audio_s': round(total_audio, 1), **agg,
        'infer_s': round(el, 2), 'vram_gb': round(vram, 2),
    })

for cid, wav, ref in TIER_B:
    p = BASE_DIR / wav
    if not p.exists():
        continue
    a = read_audio(p)
    h, el, vram = measure(lambda: mdl.long(a))
    s, hn = score((BASE_DIR / ref).read_text(encoding='utf-8', errors='replace'), h)
    transcripts[f'{mdl.label}|B|{cid}'] = hn
    rows.append({
        'model': mdl.label, 'tier': 'B', 'item': cid, 'band': '16kHz',
        'audio_s': round(len(a) / SR, 1), **{k: s[k] for k in ('S', 'D', 'I', 'N')},
        'infer_s': round(el, 2), 'vram_gb': round(vram, 2),
    })

for cid, wav, ref, band in TIER_C:
    p = BASE_DIR / wav
    if not p.exists():
        continue
    a = read_audio(p)
    h, el, vram = measure(lambda: mdl.long(a))
    s, hn = score((BASE_DIR / ref).read_text(encoding='utf-8', errors='replace'), h)
    transcripts[f'{mdl.label}|C|{cid}'] = hn
    rows.append({
        'model': mdl.label, 'tier': 'C', 'item': cid, 'band': band,
        'audio_s': round(len(a) / SR, 1), **{k: s[k] for k in ('S', 'D', 'I', 'N')},
        'infer_s': round(el, 2), 'vram_gb': round(vram, 2),
    })

new_df = pd.DataFrame(rows)
if not new_df.empty:
    new_df['wer'] = (new_df.S + new_df.D + new_df.I) / new_df.N * 100
    new_df['sub_rate'] = new_df.S / new_df.N * 100
    new_df['del_rate'] = new_df.D / new_df.N * 100
    new_df['ins_rate'] = new_df.I / new_df.N * 100
    new_df['rtfx'] = new_df.audio_s / new_df.infer_s
    new_df = new_df.round(2)
    per_item_path = OUT_DIR / 'per_item.csv'
    if per_item_path.exists():
        combined = pd.concat([pd.read_csv(per_item_path), new_df], ignore_index=True)
    else:
        combined = new_df
    combined.to_csv(per_item_path, index=False)
    transcripts_path = OUT_DIR / 'transcripts.json'
    if transcripts_path.exists():
        existing_transcripts = json.loads(transcripts_path.read_text(encoding='utf-8'))
    else:
        existing_transcripts = {}
    existing_transcripts.update(transcripts)
    transcripts_path.write_text(json.dumps(existing_transcripts, indent=2) + '\n', encoding='utf-8')
    print(new_df.groupby(['model', 'tier']).apply(rollup).round(2))

del mdl
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()